# Exploratory Data Analysis — CarDD (Car Damage Detection Dataset)

**Source:** [cardd-ustc.github.io](https://cardd-ustc.github.io/)

> **Important — this dataset is *not* on Kaggle.** CarDD is distributed by USTC via a
> licensing form; after submitting the form at
> [CarDD_license.pdf](https://cardd-ustc.github.io/docs/CarDD_license.pdf) you receive a
> Google Drive download link. The release contains two parts:
> - **`CarDD_COCO`** — COCO-format JSON annotations (bounding boxes + instance
>   segmentation polygons) over 6 damage categories, matching the "Supplementary Vision
>   Datasets" description in the reference report (Section 2.1 / 3.2).
> - **`CarDD_SOD`** — binary PNG saliency masks for the salient-object-detection task.
>
> This notebook does **not** call the Kaggle API. Instead, get the data into Colab either
> by mounting Google Drive (if you've already saved the downloaded archive there) or by
> uploading the archive directly, then run the rest of the notebook — it auto-discovers
> the COCO JSON file(s) and mask folders regardless of the exact internal layout.

1. Dataset setup (Drive mount / upload + extraction)
2. Load COCO annotations into DataFrames
3. Dataset summary statistics
4. Class distribution
5. Missing value / orphan analysis
6. Duplicate analysis (exact + near-duplicate images)
7. Feature distributions (bbox area, aspect ratio, instances/image, image resolution)
8. Mask / segmentation analysis (CarDD_SOD, if present)
9. Outlier analysis
10. Correlation analysis
11. Visualizations (histograms, bar charts, scatter plots, heatmaps)
12. Sample annotated image grid
13. Summary of findings

Run top-to-bottom in Google Colab.

## 0. Environment setup

In [ ]:
!pip install -q pycocotools imagehash --upgrade

In [ ]:
import os
import io
import json
import hashlib
import warnings
from pathlib import Path
from collections import Counter, defaultdict

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as patches
import seaborn as sns
from PIL import Image
import imagehash

warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (10, 6)

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 140)

DOWNLOAD_DIR = "/content/CarDD_release"
os.makedirs(DOWNLOAD_DIR, exist_ok=True)

## 1. Get the data into Colab

Pick **one** of the two options below (run only the cell for the option you're using).

**Option A — Google Drive** (recommended if you've already downloaded the CarDD zip/rar
after filling the licensing form, and saved it into your Drive).

**Option B — Direct upload** of the archive from your local machine (slower for large
files, but works if you haven't put it in Drive).

After extraction, both options leave the data under `DOWNLOAD_DIR`.

In [ ]:
# --- OPTION A: Google Drive ---
from google.colab import drive
drive.mount('/content/drive')

# EDIT this path to point at wherever you saved the CarDD archive in your Drive
DRIVE_ARCHIVE_PATH = "/content/drive/MyDrive/CarDD_release.zip"

if os.path.exists(DRIVE_ARCHIVE_PATH):
    if DRIVE_ARCHIVE_PATH.endswith(".zip"):
        !unzip -q -o "{DRIVE_ARCHIVE_PATH}" -d {DOWNLOAD_DIR}
    elif DRIVE_ARCHIVE_PATH.endswith((".rar",)):
        !apt-get install -y unrar -qq
        !unrar x -o+ "{DRIVE_ARCHIVE_PATH}" {DOWNLOAD_DIR}
    else:
        print("Unrecognised archive extension — extract manually.")
    print("Extraction complete.")
else:
    print(f"File not found at {DRIVE_ARCHIVE_PATH} — update DRIVE_ARCHIVE_PATH or use Option B below.")

Mounted at /content/drive
File not found at /content/drive/MyDrive/CarDD_release.zip — update DRIVE_ARCHIVE_PATH or use Option B below.


In [ ]:
# --- OPTION B: Direct upload ---
from google.colab import files

uploaded = files.upload()  # select the CarDD archive from your local machine

for fname in uploaded:
    fpath = os.path.join("/content", fname)
    with open(fpath, "wb") as f:
        f.write(uploaded[fname])
    if fname.endswith(".zip"):
        !unzip -q -o "{fpath}" -d {DOWNLOAD_DIR}
    elif fname.endswith(".rar"):
        !apt-get install -y unrar -qq
        !unrar x -o+ "{fpath}" {DOWNLOAD_DIR}
    print(f"Extracted {fname}")

Saving kaggle.json to kaggle.json
Extracted kaggle.json


In [ ]:
print("Directory tree (top 3 levels):")
for root, dirs, filenames in os.walk(DOWNLOAD_DIR):
    depth = root[len(DOWNLOAD_DIR):].count(os.sep)
    if depth < 3:
        print("  " * depth + os.path.basename(root) + "/")

Directory tree (top 3 levels):
CarDD_release/


## 2. Load COCO annotations

Auto-discovers every `.json` file under `DOWNLOAD_DIR` that has the `images` /
`annotations` / `categories` keys expected of a COCO-format annotation file — this
covers `CarDD_COCO/annotations/instances_train2017.json`-style layouts without needing
to hardcode the exact split filenames.

In [ ]:
json_candidates = [str(p) for p in Path(DOWNLOAD_DIR).rglob("*.json")]

coco_files = []
for jp in json_candidates:
    try:
        with open(jp) as f:
            data = json.load(f)
        if all(k in data for k in ("images", "annotations", "categories")):
            coco_files.append(jp)
    except Exception:
        continue

print(f"COCO annotation files found: {len(coco_files)}")
for f in coco_files:
    print(" -", f)

if not coco_files:
    print("\nNo COCO JSON found yet. Double-check DOWNLOAD_DIR contains the extracted "
          "CarDD_COCO folder, or that the archive extracted correctly in Section 1.")

COCO annotation files found: 0

No COCO JSON found yet. Double-check DOWNLOAD_DIR contains the extracted CarDD_COCO folder, or that the archive extracted correctly in Section 1.


In [ ]:
def resolve_image_path(file_name, json_path):
    candidates = [
        Path(json_path).parent / file_name,
        Path(json_path).parent.parent / file_name,
        Path(json_path).parent.parent / os.path.basename(file_name),
        Path(DOWNLOAD_DIR) / file_name,
    ]
    for c in candidates:
        if c.exists():
            return str(c)
    matches = list(Path(DOWNLOAD_DIR).rglob(os.path.basename(file_name)))
    return str(matches[0]) if matches else None

all_images, all_annos, all_cats = [], [], []
for jp in coco_files:
    with open(jp) as f:
        data = json.load(f)
    split_name = Path(jp).stem

    cat_map = {c["id"]: c["name"] for c in data["categories"]}
    for c in data["categories"]:
        all_cats.append({"category_id": c["id"], "category_name": c["name"], "source_file": split_name})

    for im in data["images"]:
        resolved = resolve_image_path(im["file_name"], jp)
        all_images.append({
            "image_id": im["id"],
            "file_name": im["file_name"],
            "filepath": resolved,
            "width": im.get("width"),
            "height": im.get("height"),
            "split": split_name,
        })

    for an in data["annotations"]:
        bbox = an.get("bbox", [None, None, None, None])
        all_annos.append({
            "annotation_id": an["id"],
            "image_id": an["image_id"],
            "category_id": an["category_id"],
            "category_name": cat_map.get(an["category_id"], "unknown"),
            "bbox_x": bbox[0], "bbox_y": bbox[1], "bbox_w": bbox[2], "bbox_h": bbox[3],
            "area": an.get("area", (bbox[2] * bbox[3]) if all(bbox) else None),
            "iscrowd": an.get("iscrowd", 0),
            "has_segmentation": bool(an.get("segmentation")),
            "split": split_name,
        })

df_images = pd.DataFrame(all_images).drop_duplicates(subset=["filepath"]).reset_index(drop=True)
df_annos = pd.DataFrame(all_annos)
df_cats = pd.DataFrame(all_cats).drop_duplicates()

print(f"Images: {len(df_images)} | Annotations: {len(df_annos)} | Categories: {df_cats['category_name'].nunique()}")
df_images.head()

KeyError: 'category_name'

In [ ]:
df_annos.head()

## 3. Dataset summary statistics

In [ ]:
instances_per_image = df_annos.groupby("image_id").size()

summary = {
    "Total images": len(df_images),
    "Total annotated instances": len(df_annos),
    "Unique categories": sorted(df_cats["category_name"].unique().tolist()),
    "Splits found": sorted(df_images["split"].unique().tolist()),
    "Mean instances per image": round(instances_per_image.mean(), 2) if len(instances_per_image) else None,
    "Median instances per image": float(instances_per_image.median()) if len(instances_per_image) else None,
    "Max instances in a single image": int(instances_per_image.max()) if len(instances_per_image) else None,
    "% annotations with segmentation polygons": round(df_annos["has_segmentation"].mean() * 100, 1) if len(df_annos) else None,
}
print(json.dumps(summary, indent=2, default=str))

In [ ]:
per_split = df_images.groupby("split").size().rename("n_images").to_frame()
per_split["n_annotations"] = df_annos.groupby("split").size()
per_split

## 4. Class distribution

In [ ]:
class_counts = df_annos["category_name"].value_counts()

fig, axes = plt.subplots(1, 2, figsize=(16, 5))
sns.barplot(x=class_counts.index, y=class_counts.values, ax=axes[0], palette="viridis")
axes[0].set_title("Instance count per damage category")
axes[0].set_xlabel("Category")
axes[0].set_ylabel("Instance count")
axes[0].tick_params(axis="x", rotation=45)
for i, v in enumerate(class_counts.values):
    axes[0].text(i, v + max(class_counts.values) * 0.01, str(v), ha="center")

axes[1].pie(class_counts.values, labels=class_counts.index, autopct="%1.1f%%",
            colors=sns.color_palette("viridis", len(class_counts)))
axes[1].set_title("Category share (%)")

plt.tight_layout()
plt.show()

imbalance_ratio = class_counts.max() / class_counts.min()
print(f"Imbalance ratio (largest class / smallest class): {imbalance_ratio:.2f} : 1")

In [ ]:
# Class distribution by split, to check stratification
class_by_split = df_annos.groupby(["split", "category_name"]).size().unstack(fill_value=0)
class_by_split

## 5. Missing value / orphan analysis

Detection-data analogue of missing values: unresolved image files, orphan annotations,
images with zero annotations, and degenerate (zero-area) bounding boxes.

In [ ]:
unresolved_images = df_images[df_images["filepath"].isna() | ~df_images["filepath"].apply(lambda p: os.path.exists(p) if p else False)]
print(f"Images referenced in JSON but not found on disk: {len(unresolved_images)}")

orphan_annos = df_annos[~df_annos["image_id"].isin(df_images["image_id"])]
print(f"Orphan annotations (image_id not in images list): {len(orphan_annos)}")

images_with_zero_annos = df_images[~df_images["image_id"].isin(df_annos["image_id"])]
print(f"Images with zero annotations: {len(images_with_zero_annos)}")

degenerate_boxes = df_annos[(df_annos["bbox_w"].fillna(0) <= 0) | (df_annos["bbox_h"].fillna(0) <= 0)]
print(f"Degenerate bounding boxes (w<=0 or h<=0): {len(degenerate_boxes)}")

missing_summary = pd.DataFrame({
    "Check": ["Unresolved image files", "Orphan annotations", "Images with 0 annotations", "Degenerate bboxes"],
    "Count": [len(unresolved_images), len(orphan_annos), len(images_with_zero_annos), len(degenerate_boxes)],
})
missing_summary

In [ ]:
df_images_valid = df_images[df_images["filepath"].apply(lambda p: bool(p) and os.path.exists(p))].reset_index(drop=True)
df_annos_valid = df_annos[
    df_annos["image_id"].isin(df_images_valid["image_id"]) &
    (df_annos["bbox_w"].fillna(0) > 0) & (df_annos["bbox_h"].fillna(0) > 0)
].reset_index(drop=True)

print(f"Valid images for analysis: {len(df_images_valid)}")
print(f"Valid annotations for analysis: {len(df_annos_valid)}")

In [ ]:
def is_valid_image(path):
    try:
        with Image.open(path) as img:
            img.verify()
        return True
    except Exception:
        return False

df_images_valid["is_readable"] = df_images_valid["filepath"].apply(is_valid_image)
corrupt_images = df_images_valid[~df_images_valid["is_readable"]]
print(f"Corrupt / unreadable image files: {len(corrupt_images)}")

df_images_valid = df_images_valid[df_images_valid["is_readable"]].reset_index(drop=True)

## 6. Duplicate analysis

Exact duplicates via MD5, near-duplicates via perceptual hash (pHash) — same
methodology used across the VehiDE / severity / COCO-car-damage EDAs for
comparability.

In [ ]:
def md5_hash(path):
    with open(path, "rb") as f:
        return hashlib.md5(f.read()).hexdigest()

df_images_valid["md5"] = df_images_valid["filepath"].apply(md5_hash)

dup_counts = df_images_valid["md5"].value_counts()
exact_dup_hashes = dup_counts[dup_counts > 1]
n_exact_dup_groups = len(exact_dup_hashes)
n_exact_dup_files = int(exact_dup_hashes.sum() - n_exact_dup_groups)

print(f"Exact-duplicate groups: {n_exact_dup_groups}")
print(f"Redundant exact-duplicate files: {n_exact_dup_files}")

In [ ]:
def phash(path):
    try:
        with Image.open(path) as img:
            return imagehash.phash(img.convert("RGB"))
    except Exception:
        return None

hashes = {}
for fp in df_images_valid["filepath"]:
    h = phash(fp)
    if h is not None:
        hashes[fp] = str(h)

hash_to_files = defaultdict(list)
for fp, h in hashes.items():
    hash_to_files[h].append(fp)

near_dup_clusters = {h: files for h, files in hash_to_files.items() if len(files) > 1}
n_near_dup_clusters = len(near_dup_clusters)
n_near_dup_redundant = sum(len(files) - 1 for files in near_dup_clusters.values())

print(f"Near-duplicate clusters: {n_near_dup_clusters}")
print(f"Redundant near-duplicate files: {n_near_dup_redundant}")

dup_summary = pd.DataFrame({
    "Duplicate type": ["Exact (MD5)", "Near-duplicate (pHash)"],
    "Groups/clusters found": [n_exact_dup_groups, n_near_dup_clusters],
    "Redundant files": [n_exact_dup_files, n_near_dup_redundant],
})
dup_summary

## 7. Feature engineering

Instance-level features (bbox area, normalised area, aspect ratio) and image-level
features (resolution, file size, instances-per-image) — same schema as the VehiDE
report (Sections 3.1, 5.3-5.5).

In [ ]:
df_annos_feat = df_annos_valid.merge(
    df_images_valid[["image_id", "width", "height", "filepath"]], on="image_id", how="inner"
)
df_annos_feat["bbox_area_px"] = df_annos_feat["bbox_w"] * df_annos_feat["bbox_h"]
df_annos_feat["bbox_area_norm"] = df_annos_feat["bbox_area_px"] / (df_annos_feat["width"] * df_annos_feat["height"])
df_annos_feat["bbox_aspect_ratio"] = (df_annos_feat["bbox_w"] / df_annos_feat["bbox_h"]).round(4)

df_annos_feat[["category_name", "bbox_w", "bbox_h", "bbox_area_px", "bbox_area_norm", "bbox_aspect_ratio"]].describe().T

In [ ]:
df_images_valid["file_size_kb"] = df_images_valid["filepath"].apply(lambda p: round(os.path.getsize(p) / 1024, 2))
df_images_valid["aspect_ratio"] = (df_images_valid["width"] / df_images_valid["height"]).round(4)

instances_per_image_valid = df_annos_feat.groupby("image_id").size().rename("n_instances")
df_images_valid = df_images_valid.merge(instances_per_image_valid, on="image_id", how="left")
df_images_valid["n_instances"] = df_images_valid["n_instances"].fillna(0).astype(int)

df_images_valid[["width", "height", "aspect_ratio", "file_size_kb", "n_instances"]].describe().T

## 8. Feature distributions

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(18, 10))
axes = axes.flatten()

sns.histplot(df_images_valid["width"], bins=40, kde=True, ax=axes[0], color="steelblue")
axes[0].set_title("Image width distribution")

sns.histplot(df_images_valid["height"], bins=40, kde=True, ax=axes[1], color="steelblue")
axes[1].set_title("Image height distribution")

sns.histplot(df_images_valid["file_size_kb"], bins=40, kde=True, ax=axes[2], color="steelblue")
axes[2].set_title("Image file size (KB) distribution")

sns.histplot(df_images_valid["n_instances"], bins=range(0, int(df_images_valid["n_instances"].max()) + 2),
             ax=axes[3], color="darkorange")
axes[3].set_title("Instances per image")

sns.histplot(df_annos_feat["bbox_area_norm"], bins=40, kde=True, ax=axes[4], color="seagreen")
axes[4].set_title("Normalised bbox area distribution")

sns.histplot(df_annos_feat["bbox_aspect_ratio"].clip(0, 5), bins=40, kde=True, ax=axes[5], color="seagreen")
axes[5].set_title("Bbox aspect ratio distribution (clipped at 5)")

plt.tight_layout()
plt.show()

In [ ]:
area_bins = pd.cut(
    df_annos_feat["bbox_area_norm"],
    bins=[0, 0.02, 0.08, 1.0],
    labels=["Minor (<0.02)", "Moderate (0.02-0.08)", "Severe (>0.08)"]
)
severity_proxy = area_bins.value_counts().sort_index()
print("Severity proxy bins (same thresholds as the VehiDE report, Section 5.3):")
print(severity_proxy)
print()
print((severity_proxy / severity_proxy.sum() * 100).round(1))

## 9. Mask / segmentation analysis (CarDD_SOD)

If the archive includes the `CarDD_SOD` salient-object-detection masks (binary PNGs),
this section auto-discovers mask folders and computes per-mask coverage ratio
(foreground pixels / total pixels) as an additional severity-adjacent signal, then
checks its correlation with the COCO bbox-area proxy above. If no mask folder is found,
this section reports zero masks and can be skipped.

In [ ]:
# Heuristic: a "mask folder" contains single-channel or binary-looking PNGs whose
# filenames often overlap with image filenames in a sibling image folder.
mask_candidates = []
for root, dirs, filenames in os.walk(DOWNLOAD_DIR):
    if "mask" in root.lower() or "gt" in root.lower() or "sod" in root.lower():
        png_files = [f for f in filenames if f.lower().endswith(".png")]
        if png_files:
            mask_candidates.append((root, len(png_files)))

print("Candidate mask folders found:")
for folder, n in mask_candidates:
    print(f" - {folder}  ({n} PNGs)")

In [ ]:
mask_rows = []
for folder, _ in mask_candidates:
    for fname in os.listdir(folder):
        if not fname.lower().endswith(".png"):
            continue
        fpath = os.path.join(folder, fname)
        try:
            with Image.open(fpath) as m:
                arr = np.array(m.convert("L"))
            coverage = float((arr > 127).mean())
            mask_rows.append({"mask_path": fpath, "folder": folder, "coverage_ratio": coverage,
                               "mask_width": arr.shape[1], "mask_height": arr.shape[0]})
        except Exception:
            continue

df_masks = pd.DataFrame(mask_rows)

if len(df_masks):
    print(f"Total masks analysed: {len(df_masks)}")
    display(df_masks[["coverage_ratio", "mask_width", "mask_height"]].describe().T)

    fig, ax = plt.subplots(figsize=(8, 5))
    sns.histplot(df_masks["coverage_ratio"], bins=40, kde=True, ax=ax, color="purple")
    ax.set_title("Mask foreground coverage ratio distribution (CarDD_SOD)")
    plt.tight_layout()
    plt.show()
else:
    print("No CarDD_SOD mask folder found in this extraction — skipping mask analysis. "
          "This is expected if you only downloaded/extracted the CarDD_COCO portion.")

## 10. Outlier analysis

In [ ]:
def iqr_outliers(series):
    q1, q3 = series.quantile(0.25), series.quantile(0.75)
    iqr = q3 - q1
    lower, upper = q1 - 1.5 * iqr, q3 + 1.5 * iqr
    mask = (series < lower) | (series > upper)
    return mask, lower, upper

outlier_features = {
    "width": df_images_valid["width"],
    "height": df_images_valid["height"],
    "file_size_kb": df_images_valid["file_size_kb"],
    "n_instances": df_images_valid["n_instances"],
    "bbox_area_norm": df_annos_feat["bbox_area_norm"],
    "bbox_aspect_ratio": df_annos_feat["bbox_aspect_ratio"],
}
if len(df_masks):
    outlier_features["mask_coverage_ratio"] = df_masks["coverage_ratio"]

outlier_rows = []
for name, series in outlier_features.items():
    mask, lower, upper = iqr_outliers(series.dropna())
    outlier_rows.append({
        "feature": name,
        "lower_bound": round(lower, 4),
        "upper_bound": round(upper, 4),
        "n_outliers": int(mask.sum()),
        "pct_outliers": round(mask.mean() * 100, 2),
    })

outlier_df = pd.DataFrame(outlier_rows)
outlier_df

In [ ]:
n_feats = len(outlier_features)
ncols = 3
nrows = int(np.ceil(n_feats / ncols))
fig, axes = plt.subplots(nrows, ncols, figsize=(6 * ncols, 5 * nrows))
axes = np.array(axes).flatten()
for ax, (name, series) in zip(axes, outlier_features.items()):
    sns.boxplot(y=series, ax=ax, color="lightcoral")
    ax.set_title(f"Boxplot: {name}")
for ax in axes[n_feats:]:
    ax.axis("off")
plt.tight_layout()
plt.show()

## 11. Correlation analysis

In [ ]:
img_corr_cols = ["width", "height", "aspect_ratio", "file_size_kb", "n_instances"]
img_corr = df_images_valid[img_corr_cols].corr()

anno_corr_cols = ["bbox_w", "bbox_h", "bbox_area_px", "bbox_area_norm", "bbox_aspect_ratio"]
anno_corr = df_annos_feat[anno_corr_cols].corr()

fig, axes = plt.subplots(1, 2, figsize=(18, 7))
sns.heatmap(img_corr, annot=True, fmt=".2f", cmap="coolwarm", center=0, ax=axes[0])
axes[0].set_title("Correlation — image-level features")

sns.heatmap(anno_corr, annot=True, fmt=".2f", cmap="coolwarm", center=0, ax=axes[1])
axes[1].set_title("Correlation — annotation/bbox-level features")

plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

sns.boxplot(data=df_annos_feat, x="category_name", y="bbox_area_norm", ax=axes[0], palette="viridis")
axes[0].set_title("Normalised bbox area by category")
axes[0].tick_params(axis="x", rotation=45)

sns.boxplot(data=df_annos_feat, x="category_name", y="bbox_aspect_ratio", ax=axes[1], palette="viridis")
axes[1].set_ylim(0, 5)
axes[1].set_title("Bbox aspect ratio by category (clipped at 5)")
axes[1].tick_params(axis="x", rotation=45)

plt.tight_layout()
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(8, 6))
sns.scatterplot(data=df_images_valid, x="width", y="height", size="n_instances",
                 hue="n_instances", palette="viridis", alpha=0.6, ax=ax)
ax.set_title("Image resolution scatter, sized/coloured by instance count")
plt.tight_layout()
plt.show()

## 12. Sample annotated image grid

In [ ]:
def draw_boxes(ax, filepath, annos_for_image, cat_colors):
    img = Image.open(filepath).convert("RGB")
    ax.imshow(img)
    for _, row in annos_for_image.iterrows():
        rect = patches.Rectangle(
            (row["bbox_x"], row["bbox_y"]), row["bbox_w"], row["bbox_h"],
            linewidth=2, edgecolor=cat_colors.get(row["category_name"], "red"), facecolor="none"
        )
        ax.add_patch(rect)
        ax.text(row["bbox_x"], max(row["bbox_y"] - 4, 0), row["category_name"],
                 color="white", fontsize=8,
                 bbox=dict(facecolor=cat_colors.get(row["category_name"], "red"), pad=1, edgecolor="none"))
    ax.axis("off")

if len(df_annos_feat):
    categories = sorted(df_annos_feat["category_name"].unique())
    palette = sns.color_palette("hsv", len(categories))
    cat_colors = {c: palette[i] for i, c in enumerate(categories)}

    sample_image_ids = df_annos_feat["image_id"].drop_duplicates().sample(
        n=min(6, df_annos_feat["image_id"].nunique()), random_state=1
    )

    fig, axes = plt.subplots(2, 3, figsize=(18, 12))
    axes = axes.flatten()
    for ax, img_id in zip(axes, sample_image_ids):
        fp = df_images_valid.loc[df_images_valid["image_id"] == img_id, "filepath"].values[0]
        annos_for_image = df_annos_feat[df_annos_feat["image_id"] == img_id]
        draw_boxes(ax, fp, annos_for_image, cat_colors)
    plt.tight_layout()
    plt.show()
else:
    print("No annotated instances available to sample — check that Section 1/2 extracted the COCO JSON correctly.")

## 13. Summary of findings

In [ ]:
print("=== EDA SUMMARY ===")
print(f"Total images (raw / valid):        {len(df_images)} / {len(df_images_valid)}")
print(f"Total annotations (raw / valid):    {len(df_annos)} / {len(df_annos_valid)}")
print(f"Unresolved image files:             {len(unresolved_images)}")
print(f"Orphan annotations:                 {len(orphan_annos)}")
print(f"Images with zero annotations:       {len(images_with_zero_annos)}")
print(f"Degenerate bboxes:                  {len(degenerate_boxes)}")
print(f"Corrupt/unreadable images:          {len(corrupt_images)}")
print(f"Exact-duplicate redundant files:    {n_exact_dup_files}")
print(f"Near-duplicate redundant files:     {n_near_dup_redundant}")
print(f"Masks analysed (CarDD_SOD):         {len(df_masks)}")
print(f"Category distribution: {dict(class_counts)}")
print(f"Category imbalance ratio (max:min): {imbalance_ratio:.2f} : 1")
print()
print("Outlier counts per feature:")
print(outlier_df[["feature", "n_outliers", "pct_outliers"]])

### Notes for write-up

- CarDD is used in this project as a **supplementary segmentation source** for the
  Damage Agent (reference report, Section 2.1) — pixel-level masks for irregularly
  shaped damage (scratches, cracks) not well captured by bounding boxes alone.
- **Missing values**: unresolved image files, orphan annotations, images with zero
  annotations, and degenerate (zero-area) bounding boxes — the detection-data analogue
  of missing cells.
- **Duplicate analysis**: identical MD5/pHash methodology used across the VehiDE,
  severity, and COCO-car-damage EDAs, so counts are directly comparable — this matters
  for the cross-dataset deduplication step described in the reference report
  (Section 7.2, "4 cross-dataset near-duplicates removed between VehiDE and CarDD").
- **Mask coverage ratio** (Section 9) is CarDD's own severity-adjacent signal; compare
  its distribution against the bbox-area severity proxy (Section 8) to sanity-check
  whether the two proxies broadly agree.
- Category names from `CarDD_COCO` should be checked against the project's 6-class
  taxonomy (`dent`, `scratch`, `crack`, `broken_lamp`/`glass_shatter`, `flat_tyre`) and
  remapped if naming differs, exactly as VehiDE's 8-to-6 remap was handled
  (`configs/class_remap.json`).
